# XGBoost

In [64]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import math
import time
from datetime import timedelta
from tabulate import tabulate
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import f1_score, make_scorer, roc_auc_score, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight 
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    'duke_lesions_radiomic' : FILE_PATH / 'duke_lesions_radiomic_medsam.csv',
    'duke_lesions' : FILE_PATH / 'duke_lesions.csv',
    'ambl_lesions_radiomic' : FILE_PATH / 'ambl_lesions_radiomic_medsam.csv',
    'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv'
}

# Training Duke
 

In [65]:
def training_duke(file_path, csv_name):

    df = pd.read_csv(file_path)

    if "Patient ID" not in df.columns:
        raise ValueError(f"{csv_name} - manca 'Patient ID'")

    # =======================
    # TARGET
    # =======================
    df["ER_class"]   = pd.to_numeric(df["ER"], errors="coerce")
    df["PR_class"]   = pd.to_numeric(df["PR"], errors="coerce")
    df["HER2_class"] = pd.to_numeric(df["HER2"], errors="coerce")

    final_target_list = ["ER_class", "PR_class", "HER2_class"]
    df = df.dropna(subset=final_target_list).copy()
    for c in final_target_list:
        df[c] = df[c].astype(int)

    # rimuovo target con una sola classe
    final_target_list = [
        c for c in final_target_list if df[c].nunique() > 1
    ]
    if len(final_target_list) == 0:
        return None

    # =======================
    # FEATURES
    # =======================
    features_to_drop = [
        "Patient ID", "lesion idx", "tumor/benign",
        "GRADE", "isTN", "Breast",
        "ER", "PR", "HER2"
    ] + final_target_list

    X = df.drop(columns=features_to_drop, errors="ignore")
    y = df[final_target_list]
    groups = df["Patient ID"]

    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.mean(numeric_only=True))
    X.columns = [re.sub(r"\[|\]|<", "", c) for c in X.columns]

    # =======================
    # STRATIFIED GROUP CV
    # =======================
    """y_strat = y["HER2_class"].astype(str)
    class_counts = y_strat.value_counts()
    n_splits = min(5, class_counts.min())

    if n_splits < 2:
        raise ValueError("Troppe poche istanze per CV stratificata")


    sgkf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )
    #splits = list(sgkf.split(X, y_strat, groups))
    splits = list(sgkf.split(X, y_strat))"""

    y_strat = y["HER2_class"].values

    counts = np.bincount(y_strat)
    min_class = counts.min()
    
    n_splits = 2
    for k in [5, 4, 3, 2]:
        if min_class % k == 0:
            n_splits = k
            break

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    splits = list(skf.split(X, y_strat))



    # Debug
    """print("\n[CHECK] Distribuzione HER2 per fold")
    for k, (tr, te) in enumerate(splits):
        tr_counts = y.iloc[tr]["HER2_class"].value_counts(normalize=True)
        te_counts = y.iloc[te]["HER2_class"].value_counts(normalize=True)
        print(f"Fold {k}")
        print(" Train:", tr_counts.to_dict())
        print(" Test :", te_counts.to_dict())"""

    # =======================
    # GRID SEARCH (F1 MACRO)
    # =======================
    base_model = XGBClassifier(
        random_state=42,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        n_jobs=1
    )

    def multi_f1(y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        return np.mean([
            f1_score(
                y_true[:, i],
                y_pred[:, i],
                average="macro",
                zero_division=0
            )
            for i in range(y_true.shape[1])
        ])


    grid = GridSearchCV(
        MultiOutputClassifier(base_model),
        param_grid={
            "estimator__n_estimators": [50, 75, 100],
            "estimator__max_depth": [2, 3, 4],
            "estimator__learning_rate": [0.05, 0.1, 1]
        },
        scoring=make_scorer(multi_f1),
        cv=splits,
        n_jobs=-1,
        error_score="raise"
    )

    grid.fit(X, y)

    best_params = {
        k.replace("estimator__", ""): v
        for k, v in grid.best_params_.items()
    }

    # =======================
    # THRESHOLD OPTIMIZATION
    # =======================
    def best_threshold(y_true, y_prob):
        ts = np.linspace(0.05, 0.95, 50)
        scores = [
            f1_score(y_true, (y_prob >= t).astype(int),
                     average="macro", zero_division=0)
            for t in ts
        ]
        return ts[np.argmax(scores)]

    # =======================
    # CV EVALUATION
    # =======================
    fold_reports = []

    for tr, te in splits:
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        fold_metrics = {}

        for i, col in enumerate(final_target_list):
            pos = y_tr[col].sum()
            neg = len(y_tr) - pos
            spw = neg / max(pos, 1)

            clf = XGBClassifier(
                **best_params,
                random_state=42,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=3,
                scale_pos_weight=spw,
                n_jobs=1
            )

            clf.fit(X_tr, y_tr[col])

            p_tr = clf.predict_proba(X_tr)[:, 1]
            p_te = clf.predict_proba(X_te)[:, 1]

            t_opt = best_threshold(y_tr[col], p_tr)
            y_pred = (p_te >= t_opt).astype(int)


            print(f"\nTarget: {col}")
            print("Confusion Matrix")
            print(confusion_matrix(y_te[col], y_pred))

            print("Classification Report")
            print(classification_report(
                y_te[col],
                y_pred,
                zero_division=0
            ))


            fold_metrics[col] = {
                "f1": f1_score(
                    y_te[col], y_pred,
                    average="macro", zero_division=0
                ),
                "accuracy": accuracy_score(y_te[col], y_pred),
                "balanced_accuracy": balanced_accuracy_score(y_te[col], y_pred),
                "auc": roc_auc_score(y_te[col], p_te)
            }

        fold_reports.append(fold_metrics)

    return {
        "best_params": best_params,
        "mean_cv_f1": grid.best_score_,
        "std_cv_f1": grid.cv_results_["std_test_score"][grid.best_index_],
        "fold_reports": fold_reports,
        "targets_used": final_target_list
    }

# Training ambl

In [66]:
def training_ambl(file_path, csv_name):
    df = pd.read_csv(file_path)

    if "Patient ID" not in df.columns:
        raise ValueError(f"{csv_name} - manca 'Patient ID' necessario per Group split")

    df_validi = df.copy()

    # --- Controllo colonne target attese ---
    required_targets = ["ER [SII]", "PR [SII]", "HER2 [SII]"]
    missing = [c for c in required_targets if c not in df_validi.columns]
    if missing:
        print(f"[ERRORE] {csv_name} - mancano colonne target: {missing}")
        return None

    # --- Creo i target binari direttamente (già binari nel Duke) ---
    final_target_list = ["ER_class", "PR_class", "HER2_class"]


    df_validi["ER_class"] = (
        pd.to_numeric(df_validi["ER [SII]"], errors="coerce") >= 1
    ).astype(int)

    df_validi["PR_class"] = (
        pd.to_numeric(df_validi["PR [SII]"], errors="coerce") >= 1
    ).astype(int)

    df_validi["HER2_class"] = (
        pd.to_numeric(df_validi["HER2 [SII]"], errors="coerce") >= 3
    ).astype(int)

    df_validi = df_validi.dropna(subset=final_target_list).copy()


    # --- Tolgo target con 1 sola classe ---
    targets_da_rimuovere = []
    for col in final_target_list:
        if df_validi[col].nunique() < 2:
            print(f"[ATTENZIONE] {csv_name} - Target {col} ha una sola classe. Lo escludo.")
            targets_da_rimuovere.append(col)

    for col in targets_da_rimuovere:
        final_target_list.remove(col)

    if len(final_target_list) == 0:
        print(f"[ERRORE] {csv_name} - Nessun target valido (>=2 classi).")
        return None

    # --- Features / Target / Groups ---
    raw_target_cols = ["ER", "PR", "HER2"]

    features_to_drop = [
        "Patient ID", "lesion idx", "tumor/benign", "GRADE", "isTN", "Breast"
    ] + raw_target_cols + final_target_list

    features = df_validi.drop(columns=features_to_drop, errors="ignore")
    target = df_validi[final_target_list]
    groups = df_validi["Patient ID"]

    # Imputazione features numeriche
    features = features.apply(pd.to_numeric, errors="coerce")
    features = features.fillna(features.mean(numeric_only=True))

    # Pulizia nomi colonne
    features.columns = [re.sub(r"\[|\]|<", "", col) for col in features.columns]

    # --- StratifiedGroupKFold ---
    """if "HER2_class" in final_target_list:
        y_strat = target["HER2_class"].astype(str)
        n_pos = int(target["HER2_class"].sum())
        n_splits = 3 if n_pos < 10 else 5
    else:
        y_strat = target.astype(int).astype(str).agg("_".join, axis=1)
        n_splits = 5

    # Debug
    min_class_count = y_strat.value_counts().min()
    n_splits = min(n_splits, min_class_count)

    if n_splits < 2:
        print(f"[ERRORE] {csv_name} - troppo pochi campioni per CV stratificata")
        return None
    
    vc = y_strat.value_counts()
    rare = vc[vc < n_splits].index
    if len(rare) > 0:
        y_strat = y_strat.where(~y_strat.isin(rare), other="RARE")

    sgkf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    #splits = list(sgkf.split(features, y_strat, groups=groups))
    splits = list(sgkf.split(features, y_strat)) """

    y_strat = target["HER2_class"].values

    counts = np.bincount(y_strat)
    min_class = counts.min()

    for k in [5, 4, 3, 2]:
        if min_class % k == 0:
            n_splits = k
            break

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    splits = list(skf.split(features, y_strat))

    
    # Debug utilissimo: controlla positivi HER2 per fold
    """print("\n[CHECK] Distribuzione HER2 per fold (DUKE)")
    for k, (tr, te) in enumerate(splits):
        tr_counts = target.iloc[tr]["HER2_class"].value_counts(normalize=True)
        te_counts = target.iloc[te]["HER2_class"].value_counts(normalize=True)
        print(f"Fold {k}")
        print(" Train:", tr_counts.to_dict())
        print(" Test :", te_counts.to_dict())"""

    # Base model with essential fixed parameters
    base_model = XGBClassifier(
        random_state=42,
        n_jobs=1,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0,
        reg_lambda=1,
        scale_pos_weight=1,
    )

    multi_output_model = MultiOutputClassifier(base_model)

    iperparametri = {
        'estimator__n_estimators': [50, 75, 100],
        'estimator__max_depth': [2, 3, 4],
        'estimator__learning_rate': [0.05, 0.1, 1]
    }

    # Multi-output scorer
    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(
                f1_score(y_true[:, i], y_pred[:, i],
                         average='macro', zero_division=0)
            )
        return np.mean(scores)

    scorer = make_scorer(multi_f1_scorer)

    tot = (len(iperparametri['estimator__n_estimators'])
           * len(iperparametri['estimator__max_depth'])
           * len(iperparametri['estimator__learning_rate']))

    #print(f"\nInizio Grid Search (GRID MINIMAL: {tot} combinazioni) per: {csv_name}")

    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=splits,             
        scoring=scorer,
        n_jobs=-1,
        verbose=1,
        return_train_score=False,
        error_score='raise'
    )

    grid_search.fit(features, target)

    best_params = grid_search.best_params_
    best_score  = grid_search.best_score_

    clean_best_params = {k.replace('estimator__', ''): v for k, v in best_params.items()}

    final_model_params = {
        'random_state': 42,
        'n_jobs': 1,
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'reg_alpha': 0,
        'reg_lambda': 1,
        'scale_pos_weight': 1,
        **clean_best_params
    }


    # Metriche per FOLD e per LABEL
    fold_reports = []

    for k, (train_idx, test_idx) in enumerate(splits):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        model_clone = MultiOutputClassifier(XGBClassifier(**final_model_params))
        model_clone.fit(X_train, y_train)

        y_pred = model_clone.predict(X_test)
        y_proba_list = model_clone.predict_proba(X_test)

        # DEBUG: Fold k
        """print(f"\n[DEBUG] Fold {k}")
        for i, col in enumerate(final_target_list):
            proba_i = y_proba_list[i]
            y_true_i = y_test.iloc[:, i].values

            print(
                f"  Target: {col} | "
                f"y_true classes: {np.unique(y_true_i)} | "
                f"proba shape: {proba_i.shape}"
            )"""

        fold_metrics = {}
        for i, col in enumerate(final_target_list):
            y_true_i = y_test.iloc[:, i]
            y_pred_i = y_pred[:, i]



            print(f"\nTarget: {col}")
            print("Confusion Matrix")
            print(confusion_matrix(y_true_i, y_pred_i))

            print("Classification Report")
            print(classification_report(
                y_true_i,
                y_pred_i,
                zero_division=0
            ))


            f1 = f1_score(y_true_i, y_pred_i, average="macro", zero_division=0)
            acc = accuracy_score(y_true_i, y_pred_i)
            bal_acc = balanced_accuracy_score(y_true_i, y_pred_i)


            auc_val = np.nan
            proba_i = y_proba_list[i]

            # Caso BINARIO
            if len(np.unique(y_true_i)) == 2 and proba_i.shape[1] == 2:
                auc_val = roc_auc_score(y_true_i, proba_i[:, 1])

            # Normalmente è binario.
            # Su AMBL però alcuni fold hanno etichette strane (>2), quindi gestisco il caso
            # con AUC OVR per non rompere il calcolo.
            elif len(np.unique(y_true_i)) > 2:
                try:
                    auc_val = roc_auc_score(
                        y_true_i,
                        proba_i,
                        multi_class="ovr",
                        average="macro"
                    )
                except ValueError:
                    auc_val = np.nan
            
            fold_metrics[col] = {
                'f1': f1,
                'accuracy': acc,
                'balanced_accuracy': bal_acc,
                'auc': auc_val
            }
            
        # Debug
        """print(f"\nMetriche Fold {k}")
        for col, m in fold_metrics.items():
            auc_str = "nan" if np.isnan(m["auc"]) else f"{m['auc']:.3f}"
            print(
                f"  {col}: "
                f"F1={m['f1']:.3f} | "
                f"ACC={m['accuracy']:.3f} | "
                f"AUC={auc_str}"
            )"""

        fold_reports.append(fold_metrics)

    final_result = {
        **clean_best_params,
        'mean_score': best_score,
        'std_score': grid_search.cv_results_['std_test_score'][grid_search.best_index_],
        'fold_reports': fold_reports,
        'targets_used': final_target_list,
        "cv_results": grid_search.cv_results_       # Aggiunta per fare Debug
    }

    return final_result

# Vado a stampare il risultato in un formato leggibile

In [67]:
def print_grid_search_results(results_per_dataset, save_csv=True, output_path="XGBoost.csv"):
    print("\n" + "=" * 80)
    print(" " * 20 + "Metriche (MEDIA ± STD) per target")
    print("=" * 80)

    rows = []

    for dataset_name, best_result in results_per_dataset.items():
        if best_result is None:
            continue

        fold_reports = best_result["fold_reports"]
        target_names = best_result.get("targets_used", [])

        print(f"\n\nDataset: {dataset_name}")
        print("-" * 80)

        for target_name in target_names:

            f1_list  = np.array([fold[target_name]["f1"] for fold in fold_reports], dtype=float)
            acc_list = np.array([fold[target_name]["accuracy"] for fold in fold_reports], dtype=float)
            auc_list = np.array([fold[target_name]["auc"] for fold in fold_reports], dtype=float)
            bal_list = np.array([fold[target_name]["balanced_accuracy"] for fold in fold_reports], dtype=float)

            f1_mean,  f1_std  = np.mean(f1_list),  np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)
            bal_mean, bal_std = np.mean(bal_list), np.std(bal_list)

            valid_auc = ~np.isnan(auc_list)
            auc_mean = np.mean(auc_list[valid_auc]) if valid_auc.any() else np.nan
            auc_std  = np.std(auc_list[valid_auc])  if valid_auc.any() else np.nan

            # ===== STAMPA =====
            print(f"\nTarget: {target_name}")
            print(f"  F1-score           = {f1_mean:.3f}  ±  {f1_std:.3f}")
            print(f"  Accuracy           = {acc_mean:.3f}  ±  {acc_std:.3f}")
            print(f"  Balanced Accuracy  = {bal_mean:.3f}  ±  {bal_std:.3f}")
            print(
                f"  AUC                = {auc_mean:.3f}  ±  {auc_std:.3f}"
                if not np.isnan(auc_mean)
                else f"  AUC                = NaN     ±  NaN"
            )

            # ===== CSV =====
            rows.append({
                "dataset": dataset_name,
                "target": target_name,
                "F1-score": f"{f1_mean:.3f} ± {f1_std:.3f}",
                "Accuracy": f"{acc_mean:.3f} ± {acc_std:.3f}",
                "Balanced Accuracy": f"{bal_mean:.3f} ± {bal_std:.3f}",
                "AUC": (
                    f"{auc_mean:.3f} ± {auc_std:.3f}"
                    if not np.isnan(auc_mean)
                    else "NaN ± NaN"
                )
            })

    # ===== SALVATAGGIO FILE =====
    if save_csv and rows:
        df_out = pd.DataFrame(rows)
        output_path = Path(output_path)
        df_out.to_csv(output_path, index=False)
        print(f"\n Risultati salvati in: {output_path.resolve()}")


# Lettura dei file

In [68]:
start_time = time.time()

# Eseguo il training per tutti i dataset
results_per_dataset = {}

for name, file_path in datasets.items():

    name_lower = name.lower()

    if "ambl" in name_lower:
        print(f"\n>>> Training AMBL: {name}")
        results_per_dataset[name] = training_ambl(file_path, name)
        print_grid_search_results(results_per_dataset)
    elif "duke" in name_lower:
        print(f"\n>>> Training DUKE: {name}")
        results_per_dataset[name] = training_duke(file_path, name)
        print_grid_search_results(results_per_dataset)
    else:   
        raise ValueError(f"Dataset non riconosciuto: {name}")
    
end_time = time.time()

# Tempo totale
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



>>> Training DUKE: duke_lesions_radiomic

Target: ER_class
Confusion Matrix
[[22 45]
 [26 53]]
Classification Report
              precision    recall  f1-score   support

           0       0.46      0.33      0.38        67
           1       0.54      0.67      0.60        79

    accuracy                           0.51       146
   macro avg       0.50      0.50      0.49       146
weighted avg       0.50      0.51      0.50       146


Target: PR_class
Confusion Matrix
[[41 41]
 [25 39]]
Classification Report
              precision    recall  f1-score   support

           0       0.62      0.50      0.55        82
           1       0.49      0.61      0.54        64

    accuracy                           0.55       146
   macro avg       0.55      0.55      0.55       146
weighted avg       0.56      0.55      0.55       146


Target: HER2_class
Confusion Matrix
[[57 47]
 [27 15]]
Classification Report
              precision    recall  f1-score   support

           0       